# Week 54, Day 4: Bitmask DP

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w054d4.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
items = ["tea", "milk", "eggs", "jam"]
mask = 0b1011
print([items[i] for i in range(4) if mask & (1 << i)])
mask |= 1 << 2
mask &= ~(1 << 0)
print(bin(mask), [items[i] for i in range(4) if mask >> i & 1])
print(bin(mask).count("1"))

In [ ]:
def assign(cost):
    n = len(cost)
    INF = float("inf")
    best = [INF] * (1 << n)
    best[0] = 0
    for mask in range(1 << n):
        if best[mask] == INF:
            continue
        w = bin(mask).count("1")
        if w == n:
            continue
        for j in range(n):
            if not mask >> j & 1:
                new = mask | (1 << j)
                best[new] = min(best[new], best[mask] + cost[w][j])
    return best[(1 << n) - 1]

print(assign([[9, 2, 7, 8], [6, 4, 3, 7], [5, 8, 1, 8], [7, 6, 9, 4]]))

In [ ]:
def tsp(dist):
    n = len(dist)
    INF = float("inf")
    best = [[INF] * n for _ in range(1 << n)]
    best[1][0] = 0
    for mask in range(1 << n):
        if not mask & 1:
            continue                       # every path starts at town 0
        for last in range(n):
            here = best[mask][last]
            if here == INF:
                continue
            for nxt in range(n):
                if not mask >> nxt & 1:
                    new = mask | (1 << nxt)
                    if here + dist[last][nxt] < best[new][nxt]:
                        best[new][nxt] = here + dist[last][nxt]
    full = (1 << n) - 1
    return min(best[full][last] + dist[last][0] for last in range(n))

print(tsp([[0, 2, 9, 10], [1, 0, 6, 4], [15, 7, 0, 8], [6, 3, 12, 0]]))

In [ ]:
def submasks(mask):
    out, sub = [], mask
    while sub:
        out.append(bin(sub))
        sub = (sub - 1) & mask
    return out

print(submasks(0b1011))

> **Key idea.** A set of up to about 20 items is an integer: bit i is item i, `|` adds, `& ~` removes, `>> i & 1` tests, and counting 1s gives the size. Masks increase as items are added, so looping masks upwards fills subsets before supersets. Assignment uses best(mask), with the next worker equal to the mask's size: O(2ⁿ × n). Held-Karp solves the travelling salesperson with best(mask, last) in O(2ⁿ × n²) and rebuilds the tour from parents. Partition problems choose the last group as a submask, with `sub = (sub - 1) & mask`, in O(3ⁿ) overall.

### Exercise: Jobs for everyone

Write `assign(cost)`: `(total, jobs)`, the least total cost of giving each of n workers a different one of n jobs, where `jobs[w]` is worker w's job. Use best(mask) over the jobs given out so far, keeping a `choice[mask]` (the last job given) to rebuild the assignment.

In [ ]:
def assign(cost):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import itertools, random\ndef test_assign():\n    \"assignment\"\n    total, jobs = assign([[9, 2, 7, 8], [6, 4, 3, 7], [5, 8, 1, 8], [7, 6, 9, 4]])\n    assert total == 13 and jobs == [1, 0, 2, 3]\n    assert assign([]) == (0, []) and assign([[5]]) == (5, [0])\n    rng = random.Random(1)\n    for _ in range(60):\n        n = rng.randint(1, 6)\n        cost = [[rng.randint(0, 20) for _ in range(n)] for _ in range(n)]\n        best = min(sum(cost[w][p[w]] for w in range(n)) for p in itertools.permutations(range(n)))\n        total, jobs = assign(cost)\n        assert total == best == sum(cost[w][jobs[w]] for w in range(n)) and sorted(jobs) == list(range(n))\n    n = 13\n    cost = [[(w * 7 + j * 11) % 17 for j in range(n)] for w in range(n)]\n    total, jobs = assign(cost)\n    assert sorted(jobs) == list(range(n)) and total == sum(cost[w][jobs[w]] for w in range(n))")

### Exercise: Held-Karp

Write `shortest_tour(dist)`: `(length, tour)` for the shortest round trip from town 0 through every town and back, with the tour as a list starting and ending at 0, by Held-Karp with a parent table. Then `shortest_path_any(dist)`: the shortest path visiting every town exactly once, starting and ending **anywhere** (no return). (Start every town's single-town mask at 0.)

In [ ]:
def shortest_tour(dist):
    ___

def shortest_path_any(dist):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import itertools, random, time\ndef test_tsp():\n    \"Held-Karp\"\n    d = [[0, 2, 9, 10], [1, 0, 6, 4], [15, 7, 0, 8], [6, 3, 12, 0]]\n    length, tour = shortest_tour(d)\n    assert length == 21 and tour[0] == tour[-1] == 0 and sorted(tour[:-1]) == [0, 1, 2, 3]\n    assert sum(d[a][b] for a, b in zip(tour, tour[1:])) == 21\n    assert shortest_tour([[0]]) == (0, [0, 0]) and shortest_path_any([[0]]) == 0\n    rng = random.Random(2)\n    for _ in range(40):\n        n = rng.randint(2, 7)\n        d = [[0 if i == j else rng.randint(1, 30) for j in range(n)] for i in range(n)]\n        tours = [sum(d[a][b] for a, b in zip((0,) + p + (0,), p + (0,))) for p in itertools.permutations(range(1, n))]\n        paths = [sum(d[a][b] for a, b in zip(p, p[1:])) for p in itertools.permutations(range(n))]\n        length, tour = shortest_tour(d)\n        assert length == min(tours) == sum(d[a][b] for a, b in zip(tour, tour[1:]))\n        assert shortest_path_any(d) == min(paths)\n    n = 12\n    pts = [(rng.randint(0, 100), rng.randint(0, 100)) for _ in range(n)]\n    d = [[abs(a[0] - b[0]) + abs(a[1] - b[1]) for b in pts] for a in pts]\n    t0 = time.perf_counter()\n    length, tour = shortest_tour(d)\n    assert sum(d[a][b] for a, b in zip(tour, tour[1:])) == length and time.perf_counter() - t0 < 10")

### Exercise: The fewest lift trips

People with given weights must go up in a lift that carries at most `capacity` kilograms (no one is heavier than that). Write `fewest_trips(weights, capacity)` with a bitmask DP: best(mask) is the pair `(trips, load of the last trip)` for carrying the people in `mask`, minimised as a pair (fewer trips, then a lighter last trip). Adding person i either fits in the last trip or starts a new one.

In [ ]:
def fewest_trips(weights, capacity):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import itertools, random\ndef brute(ws, cap):\n    n = len(ws)\n    for k in range(1, n + 1):\n        for labels in itertools.product(range(k), repeat=n):\n            loads = [0] * k\n            for w, l in zip(ws, labels):\n                loads[l] += w\n            if max(loads) <= cap:\n                return k\n    return 0\ndef test_trips():\n    \"lift\"\n    assert fewest_trips([70, 50, 80, 30, 40], 120) == 3 and fewest_trips([], 100) == 0 and fewest_trips([60], 60) == 1\n    rng = random.Random(3)\n    for _ in range(50):\n        ws = [rng.randint(1, 10) for _ in range(rng.randint(1, 7))]\n        assert fewest_trips(ws, 10) == brute(ws, 10), ws\n    ws = [rng.randint(20, 90) for _ in range(16)]\n    assert fewest_trips(ws, 100) >= -(-sum(ws) // 100)")

### Exercise: Every room once

A house has rooms 0 to n − 1 joined by doors (`doors` are pairs, both ways). Write `tours(n, doors)`: the number of walks that visit every room exactly once (starting and ending anywhere; a walk and its reverse count separately). count(mask, last) adds up the ways to visit `mask` ending at `last`.

In [ ]:
def tours(n, doors):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import itertools, random\ndef brute(n, doors):\n    adj = {(a, b) for a, b in doors} | {(b, a) for a, b in doors}\n    return sum(1 for p in itertools.permutations(range(n)) if all((x, y) in adj for x, y in zip(p, p[1:])))\ndef test_tours():\n    \"Hamiltonian paths\"\n    assert tours(3, [(0, 1), (1, 2)]) == 2 and tours(3, [(0, 1), (1, 2), (0, 2)]) == 6 and tours(1, []) == 1\n    assert tours(4, [(0, 1), (2, 3)]) == 0\n    rng = random.Random(4)\n    for _ in range(50):\n        n = rng.randint(1, 6)\n        doors = [(a, b) for a in range(n) for b in range(a + 1, n) if rng.random() < 0.5]\n        assert tours(n, doors) == brute(n, doors)")

### Exercise: As few colours as possible

Week 52's timetabler searched for a colouring. Here's the exact answer by submask DP. Write `fewest_colours(n, clashes)`: the fewest colours for nodes 0 to n − 1 so no clashing pair shares one. First mark which masks are **independent** (no clash inside), then best(mask) = 1 + the minimum of best(mask without sub) over independent non-empty submasks `sub` that contain the mask's **lowest** node.

In [ ]:
def fewest_colours(n, clashes):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import itertools, random\ndef brute(n, clashes):\n    for k in range(1, n + 1):\n        for cols in itertools.product(range(k), repeat=n):\n            if all(cols[a] != cols[b] for a, b in clashes):\n                return k\n    return 0\ndef test_colours():\n    \"chromatic number\"\n    assert fewest_colours(0, []) == 0 and fewest_colours(3, []) == 1\n    assert fewest_colours(3, [(0, 1), (1, 2), (0, 2)]) == 3\n    assert fewest_colours(5, [(i, (i + 1) % 5) for i in range(5)]) == 3\n    rng = random.Random(5)\n    for _ in range(50):\n        n = rng.randint(1, 7)\n        clashes = [(a, b) for a in range(n) for b in range(a + 1, n) if rng.random() < 0.4]\n        assert fewest_colours(n, clashes) == brute(n, clashes)\n    n = 12\n    clashes = [(a, b) for a in range(n) for b in range(a + 1, n) if (a * b + a + b) % 3 == 0]\n    assert 1 <= fewest_colours(n, clashes) <= n")

### Bug hunt: Testing the wrong bit

`mask & i` tests whether the **bits of the number i** overlap the mask, not whether item i is in it. For the mask `0b0110` (items 1 and 2) this prints `[2, 3]` instead of `[1, 2]`. Test with `1 << i`.

In [ ]:
mask = 0b0110
print([i for i in range(4) if mask & i])

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"one bit\"\n    expect_output(\"[1, 2]\")")

### Bug hunt: A tour that never goes home

This Held-Karp takes the shortest path through every town but forgets the last leg back to town 0, printing `16` instead of the tour length `21`. Add `dist[last][0]` when taking the minimum.

In [ ]:
dist = [[0, 2, 9, 10], [1, 0, 6, 4], [15, 7, 0, 8], [6, 3, 12, 0]]
n = 4
INF = float("inf")
best = [[INF] * n for _ in range(1 << n)]
best[1][0] = 0
for mask in range(1 << n):
    for last in range(n):
        if best[mask][last] == INF:
            continue
        for nxt in range(n):
            if not mask >> nxt & 1:
                new = mask | (1 << nxt)
                best[new][nxt] = min(best[new][nxt], best[mask][last] + dist[last][nxt])
full = (1 << n) - 1
print(min(best[full][last] for last in range(1, n)))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"back home\"\n    expect_output(\"21\")")

### Bug hunt: Counting down through every number

This lists submasks of `0b101` by counting down with `sub -= 1`, so it includes `0b11` and `0b10`, which use an item **not** in the mask. It prints 5 numbers; use `sub = (sub - 1) & mask` so it prints only the three submasks, `['0b101', '0b100', '0b1']`.

In [ ]:
mask = 0b101
out, sub = [], mask
while sub:
    out.append(bin(sub))
    sub -= 1
print(out)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"submasks only\"\n    expect_output(\"['0b101', '0b100', '0b1']\")")

### Puzzle: Assignment by masks

Arrange the lines of the assignment DP's main loop. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
p = ___   # the mask for the set {0, 2, 3}, as a whole number
q = ___   # how many subsets 5 items have
r = ___   # bin(0b1101 & ~(1 << 2))
s = ___   # how many non-empty submasks 0b1011 has
t = ___   # True or False: if A is a subset of B, then A <= B as numbers

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p == 13 and q == 32 and r == \"0b1001\", \"p, q and r\"\n    assert s == 7 and t is True, \"s and t\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: the shortest superstring

Fragments of a DNA sequence (or of a torn-up message) overlap. Write `shortest_superstring(words)`: one shortest string containing every word as a substring. (It's NP-hard, and important in genome assembly; for a few words, bitmask DP is exact.)

- First drop any word that is a substring of another (and duplicates).
- `overlap(a, b)`: the length of the longest suffix of a that is a prefix of b.
- Then DP like Held-Karp: best(mask, last) = the shortest length of a string containing the words in `mask` and ending with word `last`; appending word j adds `len(j) − overlap(last, j)`. Rebuild the order with parents and stitch the words together.

Also write `superstring_length(words)`. The checks compare with trying every order on small inputs and run 12 words.

In [ ]:
def overlap(a, b):
    ___

def shortest_superstring(words):
    ___

def superstring_length(words):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import itertools, random, time\ndef brute(words):\n    words = list(dict.fromkeys(words))\n    words = [w for w in words if not any(w != x and w in x for x in words)]\n    best = None\n    for p in itertools.permutations(words):\n        s = p[0] if p else \"\"\n        for w in p[1:]:\n            k = min(len(s), len(w))\n            while k and not s.endswith(w[:k]):\n                k -= 1\n            s += w[k:]\n        best = len(s) if best is None else min(best, len(s))\n    return best or 0\ndef test_super():\n    \"superstrings\"\n    assert overlap(\"abcde\", \"cdefg\") == 3 and overlap(\"abc\", \"xyz\") == 0 and overlap(\"aaa\", \"aa\") == 2\n    s = shortest_superstring([\"catg\", \"ctaagt\", \"gcta\", \"ttca\", \"atgcatc\"])\n    assert len(s) == 16 and all(w in s for w in [\"catg\", \"ctaagt\", \"gcta\", \"ttca\", \"atgcatc\"])\n    assert shortest_superstring([\"ab\", \"abc\", \"b\"]) == \"abc\" and shortest_superstring([]) == \"\"\n    rng = random.Random(6)\n    for _ in range(60):\n        words = [\"\".join(rng.choice(\"ab\") for _ in range(rng.randint(1, 4))) for _ in range(rng.randint(1, 5))]\n        s = shortest_superstring(words)\n        assert all(w in s for w in words) and len(s) == brute(words) == superstring_length(words), (words, s)\n    text = \"\".join(rng.choice(\"ACGT\") for _ in range(120))\n    frags = [text[i:i + 15] for i in range(0, 110, 9)]\n    t0 = time.perf_counter()\n    s = shortest_superstring(frags)\n    assert all(f in s for f in frags) and len(s) <= len(text) and time.perf_counter() - t0 < 10")

## Warm-up

A mask is an integer whose binary digits say which items are in the set: bit i (counting from the right, from 0) is item i. `|` adds an item, `& ~` removes one, `&` tests one, and counting the 1s gives the size.

## Sets as numbers

With n items, the subsets are exactly the integers 0 to 2ⁿ − 1: 0 is the empty set and `(1 << n) - 1` (n ones) is everything. The operations you need today:

| | Code |
|---|---|
| is item i in? | `mask >> i & 1` or `mask & (1 << i)` |
| add item i | `mask | (1 << i)` |
| remove item i | `mask & ~(1 << i)` |
| size of the set | `bin(mask).count("1")` (or `mask.bit_count()` in Python 3.10 and later) |
| everything | `(1 << n) - 1` |

A useful fact for filling tables: if A is a subset of B, then A ≤ B as numbers (removing bits only makes a number smaller). So looping `mask` from 0 upwards visits every subset before any superset, which is the fill order whenever a set's value depends on its smaller subsets.

## Who does which job?

n workers, n jobs, and `cost[w][j]` for worker w doing job j: give each worker a different job at the least total cost. Trying every assignment is n! (3.6 million for 10, 2.4 × 10¹⁸ for 20). Instead, hand out jobs to workers in order 0, 1, 2, ...:

**best(mask) = the least cost of giving the jobs in `mask` to the first `size(mask)` workers**.

The worker who gets a job next is w = size(mask); for each job j not in the mask, best(mask | 1 << j) can be best(mask) + cost[w][j]. That's 2ⁿ masks with n choices each: O(2ⁿ × n), about 20 million steps for n = 20. (The Hungarian algorithm, from Harold Kuhn in 1955, solves assignment in polynomial time; the bitmask DP is simpler and the pattern generalises.)

## The travelling salesperson

Visit every town once and return home at the least total distance. Week 52 pruned the n! tours with branch and bound. The **Held-Karp** algorithm (Richard Bellman, and Michael Held with Richard Karp, both 1962) uses the fact that once you know which towns you've visited and where you are now, the order you visited them in no longer matters:

**best(mask, last) = the shortest path that starts at town 0, visits exactly the towns in `mask`, and ends at `last`**,

with best({0}, 0) = 0. Extend by going from `last` to any unvisited town. The answer is the minimum over `last` of best(everything, last) + dist[last][0]. There are 2ⁿ × n states and n choices each: O(2ⁿ × n²). For 15 towns that's about 7 million steps, against 14!/2 ≈ 44 billion tours. To rebuild the tour, keep `parent[mask][last]`, the town before `last`.

It's still exponential (the problem is NP-hard), but exponential in a much gentler way, and it's the fastest known exact method in the worst case.

## Submasks

Partition problems ask how to split a set into groups: the fewest lift trips, the fewest colours so no two neighbours share one, the best way to divide tasks into shifts. Then

**best(mask) = the best way to deal with the items in `mask`**,

and the last group is some non-empty **submask** of `mask`. The loop `sub = (sub - 1) & mask` visits every submask of `mask`, from `mask` itself down to (but not including) 0, skipping all the numbers that aren't submasks. Over all masks, the total work is 3ⁿ (each item is outside the mask, in the group, or in the mask but not the group): about 3.5 million for n = 14, 3.5 billion for n = 20. So submask DPs suit n up to about 15.

A common trick halves the work: insist that the group contains the **lowest** item of the mask, so each partition is built in only one order.

## How small is small?

| Method | Work | Comfortable n in Python |
|---|---|---|
| Try every order | n! | about 10 |
| Bitmask DP over sets | 2ⁿ × n | about 20 |
| Held-Karp | 2ⁿ × n² | about 16 |
| Submask DP | 3ⁿ | about 14 |
| Try every subset | 2ⁿ × n | about 20 |

The tell-tale sign of a bitmask DP in a problem statement is a small bound on n (often 15 to 20) together with "every", "each exactly once" or "split into groups". If n is in the thousands, the problem wants something else.

## Practice

## Quiz

## Daily challenge

## Before you go

Pick five places you visit in a week and estimate the distances between them; how many possible round trips are there, and how many states would Held-Karp use? Tomorrow: DP practice, choosing the technique for a mixed set of problems.